# Unit 5: 频域分析与特征提取

## 学习目标
- 理解 FFT 和功率谱密度（PSD）
- 掌握各频段（δ, θ, α, β, γ）的功率计算
- 学习统计特征（均值、标准差、RMS）
- 构建特征向量用于后续 ML 分析
- 对比不同通道的频谱特征

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from brainflow.board_shim import BoardShim, BrainFlowInputParams, BoardIds
from brainflow.data_filter import (
    DataFilter, FilterTypes, AggOperations,
    DetrendOperations, WindowOperations, NoiseTypes
)

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("模块导入完成")

## 5.1 准备数据并预处理

先采集数据并执行标准预处理流水线。

In [ ]:
# 采集数据
board = BoardShim(BoardIds.SYNTHETIC_BOARD, BrainFlowInputParams())
board.prepare_session()
board.start_stream()
time.sleep(15)
data = board.get_board_data()
board.stop_stream()
board.release_session()

descr = BoardShim.get_board_descr(BoardIds.SYNTHETIC_BOARD)
eeg_channels = descr['eeg_channels']
eeg_names = BoardShim.get_eeg_names(BoardIds.SYNTHETIC_BOARD)
sampling_rate = descr['sampling_rate']

print(f"采集了 {data.shape[1]} 个样本，{sampling_rate} Hz，约 {data.shape[1]/sampling_rate:.1f} 秒")

# 标准预处理
def preprocess_eeg(data, ch_idx, sr):
    signal = data[ch_idx].copy().astype(np.float64)
    DataFilter.detrend(signal, DetrendOperations.LINEAR.value)
    DataFilter.perform_bandpass(signal, sr, 22.5, 45.0, 4,
                                 FilterTypes.BUTTERWORTH.value, 0)
    DataFilter.perform_bandstop(signal, sr, 50.0, 4.0, 4,
                                 FilterTypes.BUTTERWORTH.value, 0)
    return signal

eeg_data = data[eeg_channels, :].copy().astype(np.float64)
processed_eeg = np.array([preprocess_eeg(data, ch, sampling_rate) 
                           for ch in eeg_channels])
print(f"预处理完成，形状: {processed_eeg.shape}")

## 5.2 FFT 与 PSD 基础

**FFT（快速傅里叶变换）**：将时域信号转换到频域。

**PSD（功率谱密度）**：描述信号功率在频率上的分布。BrainFlow 支持 Welch 方法。

`DataFilter.get_psd_welch(data, nfft, overlap, sampling_rate, window)`
- 返回 `(amplitudes, frequencies)` 两个数组

In [ ]:
# 计算 Fz 通道的 PSD
ch_data = processed_eeg[0, :]

# Welch 方法计算 PSD
nfft = DataFilter.get_nearest_power_of_two(sampling_rate)
psd_ampl, psd_freq = DataFilter.get_psd_welch(
    ch_data, nfft, nfft // 2, sampling_rate,
    WindowOperations.BLACKMAN_HARRIS.value
)

print(f"NFFT: {nfft}")
print(f"PSD 振幅长度: {len(psd_ampl)}")
print(f"频率分辨率: {psd_freq[1] - psd_freq[0]:.3f} Hz")

# 绘制 PSD
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(psd_freq[:len(psd_freq)//2], psd_ampl[:len(psd_ampl)//2])
ax.set_xlabel('频率 (Hz)')
ax.set_ylabel('功率/频率 (dB/Hz)')
ax.set_title('Fz 通道 — 功率谱密度 (PSD)')
ax.grid(True, alpha=0.3)

# 标注频带
bands = [
    (0.5, 4, 'δ (Delta)', 'purple'),
    (4, 8, 'θ (Theta)', 'blue'),
    (8, 13, 'α (Alpha)', 'green'),
    (13, 30, 'β (Beta)', 'orange'),
    (30, 50, 'γ (Gamma)', 'red'),
]
for low, high, name, color in bands:
    ax.axvspan(low, high, alpha=0.15, color=color, label=name)
ax.legend(loc='upper right', fontsize=8)
ax.set_xlim(0, 60)
plt.tight_layout()
plt.show()

## 5.3 各通道 PSD 对比

不同脑区的 EEG 信号有不同的频谱特征。

In [ ]:
# 绘制所有 EEG 通道的 PSD 对比
fig, axes = plt.subplots(4, 2, figsize=(14, 10))
axes = axes.flatten()

nfft = DataFilter.get_nearest_power_of_two(sampling_rate)

for i in range(8):
    ampl, freq = DataFilter.get_psd_welch(
        processed_eeg[i, :], nfft, nfft // 2, sampling_rate,
        WindowOperations.BLACKMAN_HARRIS.value
    )
    half = len(freq) // 2
    axes[i].plot(freq[:half], ampl[:half], linewidth=0.8)
    axes[i].set_title(f'PSD - {eeg_names[i]}')
    axes[i].set_xlim(0, 60)
    axes[i].grid(True, alpha=0.3)
    if i >= 6:
        axes[i].set_xlabel('频率 (Hz)')

plt.suptitle('8 通道 EEG 功率谱密度对比', fontsize=14)
plt.tight_layout()
plt.show()

## 5.4 频带功率计算 —— 核心特征

**频带功率是 BCI 中最常用的特征之一。** BrainFlow 提供了两种计算方式：

### 方式1: 对单个通道通过 PSD 计算频带功率

1. `DataFilter.get_psd_welch(data, nfft, overlap, sr, window)` — 计算 PSD
2. `DataFilter.get_band_power(psd_ampl, psd_freq, freq_start, freq_end)` — 从 PSD 提取频带功率

### 方式2: 对所有通道一次计算多个频带（推荐）

`DataFilter.get_avg_band_powers(eeg_data_slice, eeg_channels, sampling_rate, apply_filters)`

返回 `(band_powers, stddevs)` — 每个元素是 5 个频带的平均值

In [ ]:
# ===== 方式1: 通过 PSD 逐频带计算（Fz 通道）=====
ch_data = processed_eeg[0, :]

band_ranges = [
    ("Delta (δ)", 0.5, 4),
    ("Theta (θ)", 4, 8),
    ("Alpha (α)", 8, 13),
    ("Beta (β)",  13, 30),
    ("Gamma (γ)", 30, 50),
]

# 计算 PSD
nfft_band = DataFilter.get_nearest_power_of_two(sampling_rate)
psd = DataFilter.get_psd_welch(ch_data, nfft_band, nfft_band // 2, sampling_rate,
                                 WindowOperations.BLACKMAN_HARRIS.value)

print(f"Fz 通道频带功率:")
print(f"{'频带':<15} {'功率':<12}")
print("-" * 30)

powers = []
for name, low, high in band_ranges:
    power = DataFilter.get_band_power(psd[0], psd[1], low, high)
    powers.append(power)
    print(f"{name:<15} {power:<12.4f}")

total_power = sum(powers)
print(f"{'总计':<15} {total_power:<12.4f}")

# 饼图展示各频带占比
fig, ax = plt.subplots(figsize=(6, 6))
colors = ['purple', 'blue', 'green', 'orange', 'red']
labels = [b[0] for b in band_ranges]
ax.pie(powers, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax.set_title('Fz — 各频带能量占比')
plt.show()

In [ ]:
# ===== 方式2: get_avg_band_powers 批量计算（推荐）=====
# 取一个时间窗口（比如 4 秒的窗口）
window_size = sampling_rate * 4  # 4秒
if processed_eeg.shape[1] >= window_size:
    eeg_window = processed_eeg[:, :window_size]
else:
    eeg_window = processed_eeg

print(f"分析窗口: {window_size} 样本 ({window_size/sampling_rate:.0f} 秒)")

# get_avg_band_powers 一次性计算所有通道的所有频带
# 参数: (data_2d, eeg_channel_indices, sampling_rate, apply_filters)
result = DataFilter.get_avg_band_powers(
    eeg_window,
    list(range(len(eeg_channels))),  # 输入通道的本地索引 [0,1,2,...,7]
    sampling_rate,
    True  # 自动应用内部滤波器
)

# 结果: (avg_band_powers, stddev_band_powers)
# avg_band_powers[0] = 所有通道 Delta 的平均功率
# avg_band_powers[1] = 所有通道 Theta 的平均功率
# ...等
avg_band_powers = result[0]
stddev_band_powers = result[1]

print(f"\n所有通道平均频带功率:")
band_names = ['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma']
for i, name in enumerate(band_names):
    print(f"  {name:<8}: {avg_band_powers[i]:.4f} ± {stddev_band_powers[i]:.4f}")

# 可视化
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(band_names))
bars = ax.bar(x, avg_band_powers, yerr=stddev_band_powers, 
              color=colors, capsize=5, alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(band_names)
ax.set_ylabel('平均功率')
ax.set_title('全通道平均频带功率 (4秒窗口)')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 5.5 统计特征

除了频带功率，时域统计特征也是重要的信号表征。

In [ ]:
# 计算 Fz 通道的统计特征
ch_data = processed_eeg[0, :]

mean_val = DataFilter.get_mean(ch_data)
std_val = DataFilter.get_standard_deviation(ch_data)
rms_val = DataFilter.get_root_mean_square(ch_data)

print(f"Fz 通道统计特征:")
print(f"  均值 (Mean):         {mean_val:.4f} μV")
print(f"  标准差 (Std):        {std_val:.4f} μV")
print(f"  均方根 (RMS):        {rms_val:.4f} μV")

# 所有通道的统计特征
print(f"\n所有通道统计:")
print(f"{'通道':<8} {'均值':<12} {'标准差':<12} {'RMS':<12}")
print("-" * 45)
for i in range(len(eeg_channels)):
    ch_data = processed_eeg[i, :]
    m = DataFilter.get_mean(ch_data)
    s = DataFilter.get_standard_deviation(ch_data)
    r = DataFilter.get_root_mean_square(ch_data)
    print(f"{eeg_names[i]:<8} {m:<12.4f} {s:<12.4f} {r:<12.4f}")

## 5.6 构建特征向量

将时域和频域特征组合成一个特征向量，这是 ML 模型的输入。

### 典型特征向量结构

```
[ch1_delta, ch1_theta, ch1_alpha, ch1_beta, ch1_gamma,  # 通道1频带功率
 ch2_delta, ch2_theta, ch2_alpha, ch2_beta, ch2_gamma,  # 通道2频带功率
 ...
 ch1_mean, ch1_std, ch1_rms,                            # 通道1统计特征
 ...]
```

In [ ]:
def build_feature_vector(eeg_data_2d, sampling_rate, channel_names=None):
    """
    从多通道 EEG 数据构建特征向量。
    
    Args:
        eeg_data_2d: (n_channels, n_samples) 的 EEG 数据数组
        sampling_rate: 采样率 (Hz)
        channel_names: 通道名列表（可选）
    
    Returns:
        feature_vector: 一维特征数组
        feature_labels: 特征标签列表
    """
    n_channels = eeg_data_2d.shape[0]
    feature_vector = []
    feature_labels = []
    
    band_ranges = [
        ("Delta", 0.5, 4),
        ("Theta", 4, 8),
        ("Alpha", 8, 13),
        ("Beta", 13, 30),
        ("Gamma", 30, 50),
    ]
    
    for ch_idx in range(n_channels):
        ch_name = channel_names[ch_idx] if channel_names else f"Ch{ch_idx+1}"
        ch_data = eeg_data_2d[ch_idx, :]
        
        # 计算 PSD（每个通道一次）
        nfft = DataFilter.get_nearest_power_of_two(sampling_rate)
        psd = DataFilter.get_psd_welch(ch_data, nfft, nfft // 2, sampling_rate,
                                         WindowOperations.BLACKMAN_HARRIS.value)
        
        # 从 PSD 提取频带功率特征
        for band_name, low, high in band_ranges:
            power = DataFilter.get_band_power(psd[0], psd[1], low, high)
            feature_vector.append(power)
            feature_labels.append(f"{ch_name}_{band_name}")
        
        # 统计特征
        mean_val = DataFilter.get_mean(ch_data)
        std_val = DataFilter.get_standard_deviation(ch_data)
        rms_val = DataFilter.get_root_mean_square(ch_data)
        
        feature_vector.extend([mean_val, std_val, rms_val])
        feature_labels.extend([f"{ch_name}_Mean", f"{ch_name}_Std", f"{ch_name}_RMS"])
    
    return np.array(feature_vector, dtype=np.float64), feature_labels

# 构建特征向量
feature_vector, feature_labels = build_feature_vector(
    processed_eeg, sampling_rate, eeg_names
)

print(f"特征向量长度: {len(feature_vector)}")
print(f"特征构成: {len(eeg_channels)} 通道 × (5 频带 + 3 统计) = {len(eeg_channels) * 8} 维")
print(f"\n特征预览:")
for i in range(0, min(len(feature_labels), 16)):
    print(f"  {feature_labels[i]:<20} = {feature_vector[i]:.4f}")

## 5.7 滑动窗口特征提取

在实时 BCI 中，需要在滑动窗口上持续提取特征。这是时间动态分析的基础。

In [ ]:
def sliding_window_features(eeg_data_2d, sampling_rate, window_sec=2, step_sec=0.5):
    """
    滑动窗口特征提取。
    
    Args:
        eeg_data_2d: (n_channels, n_samples)
        sampling_rate: 采样率
        window_sec: 窗口大小（秒）
        step_sec: 步长（秒）
    
    Returns:
        features_2d: (n_windows, n_features)
        timestamps: 每个窗口的中心时间索引
    """
    window_samples = int(window_sec * sampling_rate)
    step_samples = int(step_sec * sampling_rate)
    n_total = eeg_data_2d.shape[1]
    
    all_features = []
    all_timestamps = []
    
    for start in range(0, n_total - window_samples, step_samples):
        end = start + window_samples
        window_data = eeg_data_2d[:, start:end]
        
        # 对窗口内每个通道计算 PSD 后提取频带功率
        window_features = []
        nfft = DataFilter.get_nearest_power_of_two(sampling_rate)
        for ch_idx in range(window_data.shape[0]):
            ch_data = window_data[ch_idx, :]
            psd = DataFilter.get_psd_welch(ch_data, nfft, nfft // 2, sampling_rate,
                                             WindowOperations.BLACKMAN_HARRIS.value)
            alpha_power = DataFilter.get_band_power(psd[0], psd[1], 8.0, 13.0)
            beta_power = DataFilter.get_band_power(psd[0], psd[1], 13.0, 30.0)
            window_features.extend([alpha_power, beta_power])
        
        all_features.append(window_features)
        all_timestamps.append(start + window_samples // 2)
    
    return np.array(all_features), np.array(all_timestamps)

# 演示滑动窗口
window_features, window_times = sliding_window_features(
    processed_eeg, sampling_rate, window_sec=2, step_sec=0.5
)

print(f"窗口数: {window_features.shape[0]}")
print(f"每窗口特征数: {window_features.shape[1]}")

# 可视化 Fz 通道的 Alpha 功率随时间变化
fz_alpha_idx = 0  # 第一个通道的第一个频带就是 Fz Alpha
fz_beta_idx = 1   # Fz Beta

fig, ax = plt.subplots(figsize=(14, 4))
time_axis = window_times / sampling_rate
ax.plot(time_axis, window_features[:, fz_alpha_idx], label='Fz Alpha (8-13Hz)', linewidth=1.5)
ax.plot(time_axis, window_features[:, fz_beta_idx], label='Fz Beta (13-30Hz)', linewidth=1.5)
ax.set_xlabel('时间 (秒)')
ax.set_ylabel('频带功率')
ax.set_title('滑动窗口 — Fz 通道 Alpha/Beta 功率时间演变')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5.8 时间-频率热力图

一种直观展示频域特征随时间演变的方式。

In [ ]:
# 为 Fz 通道构建时间-频率热力图
fz_data = processed_eeg[0, :]

nfft = 256
noverlap = nfft // 2
step = nfft - noverlap

spectrogram = []
time_positions = []

for start in range(0, len(fz_data) - nfft, step):
    segment = fz_data[start:start + nfft]
    ampl, freq = DataFilter.get_psd_welch(
        segment, nfft, nfft // 2, sampling_rate,
        WindowOperations.HANNING.value
    )
    spectrogram.append(ampl[:60])  # 只保留 0-60 Hz
    time_positions.append((start + nfft // 2) / sampling_rate)

spectrogram = np.array(spectrogram).T  # (freq, time)

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.pcolormesh(time_positions, freq[:60], 
                    10 * np.log10(spectrogram + 1e-10),
                    shading='auto', cmap='viridis')
ax.set_xlabel('时间 (秒)')
ax.set_ylabel('频率 (Hz)')
ax.set_title('Fz 通道 — 时频热力图 (Spectrogram)')
fig.colorbar(im, ax=ax, label='功率 (dB)')

# 标注频带
for low, high, name, color in bands:
    ax.axhline(y=low, color=color, linestyle='--', alpha=0.6, linewidth=0.8)
    ax.axhline(y=high, color=color, linestyle='--', alpha=0.6, linewidth=0.8)
    ax.text(time_positions[0], (low + high) / 2, name, fontsize=7,
            va='center', ha='left', fontweight='bold')

plt.tight_layout()
plt.show()

## 小结

### 特征提取速查

| 类别 | 方法 | 说明 |
|------|------|------|
| PSD | `get_psd_welch(data, nfft, overlap, sr, window)` | 功率谱密度 |
| 单频带功率 | `get_psd_welch()` + `get_band_power()` | PSD → 频带功率 |
| 多频带功率 | `get_avg_band_powers(data, channels, sr, filters)` | 所有通道×所有频带 |
| 均值 | `get_mean(data)` | 时域统计 |
| 标准差 | `get_standard_deviation(data)` | 时域统计 |
| RMS | `get_root_mean_square(data)` | 时域统计 |
| 滑动窗口 | 自定义循环 | 实时特征提取 |

### 典型特征向量

**8 通道 × (5 频带 + 3 统计) = 64 维特征向量**

→ [Unit 6: ML 集成](unit6_ml_integration.ipynb) — 使用 BrainFlow 内置 ML 模型进行心理状态评估